# `bz-ggml`: Turnkey Dual Tesla T4 Production Engine & Studio on Kaggle

High-performance C++ / CUDA inference engine for **Breeze-TTS-2** powered by **GGML** on Dual NVIDIA Tesla T4 GPUs.

### Turnkey Highlights (Zero Runtime Patches Needed)
- **Auto Precision Sizing**: Automatically detects **FP16** (`19,500` tokens/GPU) vs **Q8_0** (`33,100`/`24,400` tokens/GPU) directly from the `.gguf` filename.
- **Instant Startup**: Uses precompiled CUDA `sm_75` binaries (`breeze-cuda-sm75.tar.gz`) from `prebuilt-t4` (`< 15s` setup, zero CMake compilation).
- **Option A Locked Consistency**: Coherent `45–60w` cloning chunks, `depth_temperature=0.0`, `depth_cfg_codebooks=3`, isolated per-chunk vocoding with bounded RMS leveling & `200 ms` breath pauses, and auto-SNR enhancer gating (`>= 30 dB`).
- **Three Turnkey Run Modes**:
  1. **Headless E2E Sanity Check**: Pure dual-GPU cluster inference + Whisper Turbo ASR verification.
  2. **Production mTLS Worker Gateway**: Persistent worker node streaming 1.0s telemetry & executing cloud jobs via `AdmissionController` + `MTLSGatewayClient`.
  3. **Gradio Studio UI**: Updated interactive studio (`Option A` defaults, auto-gated `.breeze` registration, long-form synthesis, Mode 4 conversion, and subtitles).

## 1. Hardware Verification
Confirm two NVIDIA Tesla T4 GPUs (16 GB VRAM each) are active.

In [ ]:
!nvidia-smi

## 2. Turnkey Production Setup (Dependencies + Prebuilt CUDA Binary + Model)
Clones/updates `main`, installs all verified Python 3.12 runtime dependencies cleanly, links prebuilt CUDA `sm_75` binaries into system library paths, and downloads the chosen GGUF weights.

In [ ]:
import os, subprocess, sys, time

t0 = time.time()

# ----------------------------------------------------------------------
# MODEL PRECISION SELECTOR (Uncomment the precision you want to serve)
# ----------------------------------------------------------------------
MODEL_NAME = "breeze-tts-2-f16.gguf"    # ~5.99 GB (Studio FP16 fidelity, auto 19,500 tokens/GPU)
# MODEL_NAME = "breeze-tts-2-q8_0.gguf"  # ~3.32 GB (Fast INT8 quantized, auto 24,400/33,100 tokens/GPU)
# ----------------------------------------------------------------------

print("[1/4] Cloning / updating bz-ggml (main branch)...")
if not os.path.exists("/kaggle/working/bz-ggml/.git"):
    subprocess.run("git clone -b main https://github.com/sharjeel103/bz-ggml.git /kaggle/working/bz-ggml", shell=True, check=True)
else:
    subprocess.run(
        "git -C /kaggle/working/bz-ggml fetch origin main && "
        "git -C /kaggle/working/bz-ggml checkout main && "
        "git -C /kaggle/working/bz-ggml pull --ff-only origin main",
        shell=True,
        check=True,
    )

print("[2/4] Installing verified Python 3.12 runtime dependencies...")
subprocess.run(
    "pip install -q -r /kaggle/working/bz-ggml/requirements.txt && "
    "pip install -q --no-deps resemble-enhance",
    shell=True,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-c",
        "import faster_whisper, gradio, soundfile; from resemble_enhance.enhancer.inference import load_enhancer, inference; print('All Python deps verified OK')",
    ],
    check=True,
)

print("[3/4] Installing & symlinking prebuilt CUDA sm_75 shared libraries...")
os.makedirs("/kaggle/working/bz-ggml/build", exist_ok=True)
bin_tar = "/kaggle/working/breeze-cuda-sm75.tar.gz"
prebuilt_lib_dir = "/kaggle/working/breeze-cuda-sm75/lib"
if not os.path.exists(os.path.join(prebuilt_lib_dir, "libbreeze.so")):
    BIN_URL = "https://github.com/sharjeel103/bz-ggml/raw/prebuilt-t4/breeze-cuda-sm75.tar.gz"
    subprocess.run(f"wget -q -c {BIN_URL} -O {bin_tar}", shell=True, check=True)
    subprocess.run(f"tar -xzf {bin_tar} -C /kaggle/working/", shell=True, check=True)
subprocess.run(f"cp -P {prebuilt_lib_dir}/* /kaggle/working/bz-ggml/build/", shell=True, check=True)
subprocess.run("ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/lib/x86_64-linux-gnu/libcuda.so", shell=True)
subprocess.run("rm -f /usr/lib/x86_64-linux-gnu/libbreeze*.so* /usr/lib/x86_64-linux-gnu/libggml*.so*", shell=True)
subprocess.run("ln -sf /kaggle/working/bz-ggml/build/*.so* /usr/lib/x86_64-linux-gnu/ && ldconfig", shell=True, check=True)

print(f"[4/4] Verifying Hugging Face weights ({MODEL_NAME})...")
os.makedirs("/kaggle/working/models", exist_ok=True)
model_path = os.path.join("/kaggle/working/models", MODEL_NAME)
if not os.path.exists(model_path):
    from huggingface_hub import hf_hub_download
    hf_hub_download(repo_id="HoppouAI/Breeze-TTS-2.cpp", filename=MODEL_NAME, local_dir="/kaggle/working/models")

with open("/kaggle/working/models/active_model.txt", "w") as f:
    f.write(model_path)

print()
print(f"Turnkey setup ready in {time.time() - t0:.1f}s! Active model: {MODEL_NAME}")


---
## Mode 1: Headless E2E Sanity Check (Zero Gradio)
Verifies the dual-GPU cluster, Whisper Turbo ASR microservice, and Option A TTS pipeline, then cleanly releases 100% GPU VRAM.

In [ ]:
import gc, glob, os, sys, time
sys.path.insert(0, "/kaggle/working/bz-ggml/python")
from bz_ggml import ASRService, DualInstanceCluster, EngineOrchestrator, Parcel
from bz_ggml.audio_pipeline import encode_pcm_to_wav_bytes
from IPython.display import Audio, display

model_path = None
if os.path.exists("/kaggle/working/models/active_model.txt"):
    with open("/kaggle/working/models/active_model.txt") as f:
        model_path = f.read().strip()
if not model_path or not os.path.exists(model_path):
    models = sorted(glob.glob("/kaggle/working/models/*.gguf"))
    if not models:
        raise FileNotFoundError("No .gguf model found in /kaggle/working/models. Run Setup cell first.")
    model_path = models[-1]

print(f"Initializing DualInstanceCluster + ASRService for {os.path.basename(model_path)}...")
asr_service = ASRService(model_size="turbo", device="cuda", device_index=0, compute_type="float16")
cluster = DualInstanceCluster(
    model_path=model_path,
    lib_path="/kaggle/working/bz-ggml/build/libbreeze.so",
    enable_vocoder=True,
    vocoder_mode="on",
    precision="auto",
    enable_asr=True,
    enable_enhancer=True,
)
orchestrator = EngineOrchestrator(cluster=cluster, asr_service=asr_service)

try:
    parcel = Parcel.tts_pipeline(
        script_text="The production inference engine is online across dual Tesla T4 GPUs with zero runtime patches.",
        instruction="Speak clearly and naturally.",
        enhance=False,
        master=True,
        srt=True,
        parcel_id="kaggle_e2e_demo",
    )
    t0 = time.time()
    res = orchestrator.dispatch(parcel, progress_callback=lambda s, p, m: print(f"  [{s.upper():<8}] {p*100:5.1f}% | {m}"))
    elapsed = time.time() - t0
    assert res.success and res.audio_pcm is not None, res.error_message

    print(f"\nDone: {res.duration_s:.2f}s audio in {elapsed:.2f}s ({res.chunks_count} chunk(s)) | {res.stage_timings}")
    if res.transcript:
        print(f"Verbatim ASR: \"{res.transcript}\"")
    display(Audio(encode_pcm_to_wav_bytes(res.audio_pcm, res.sample_rate), rate=res.sample_rate))
finally:
    orchestrator.shutdown()
    cluster.close()
    del asr_service, orchestrator, cluster
    gc.collect()
    print("Cluster closed cleanly; 100% GPU VRAM reclaimed.")


---
## Mode 2: Production Worker Server (`MTLSGatewayClient`)
Connects this Kaggle Dual-T4 node outbound via mTLS to your central backend platform server (`AdmissionController` + 1.0s hardware telemetry + full `tts`, `voice_registration`, `enhance`, `asr` dispatch).

In [ ]:
import asyncio, glob, os, sys
sys.path.insert(0, "/kaggle/working/bz-ggml/python")
from bz_ggml import ASRService, DualInstanceCluster, EngineOrchestrator
from bz_ggml.gateway import AdmissionController, MTLSGatewayClient

# Configure your backend WebSocket endpoint & certificate paths before running this cell:
SERVER_WS_URL = os.environ.get("BREEZE_GATEWAY_URL", "wss://your-backend.example.com:9443/ws")
CA_CERT_PATH = os.environ.get("BREEZE_CA_CERT", "/kaggle/working/certs/ca.pem")
CLIENT_CERT_PATH = os.environ.get("BREEZE_CLIENT_CERT", "/kaggle/working/certs/worker.crt")
CLIENT_KEY_PATH = os.environ.get("BREEZE_CLIENT_KEY", "/kaggle/working/certs/worker.key")
REQUIRE_MTLS = os.environ.get("BREEZE_REQUIRE_MTLS", "1") != "0"

if "your-backend.example.com" in SERVER_WS_URL:
    raise RuntimeError("Set SERVER_WS_URL / BREEZE_GATEWAY_URL and cert paths to start the mTLS worker server.")

with open("/kaggle/working/models/active_model.txt") as f:
    model_path = f.read().strip()

asr_service = ASRService(model_size="turbo", device="cuda", device_index=0, compute_type="float16")
cluster = DualInstanceCluster(
    model_path=model_path,
    lib_path="/kaggle/working/bz-ggml/build/libbreeze.so",
    precision="auto",
    enable_vocoder=True,
    vocoder_mode="on",
    enable_asr=True,
    enable_enhancer=True,
)
orchestrator = EngineOrchestrator(cluster=cluster, asr_service=asr_service)
admission = AdmissionController(orchestrator=orchestrator)
gateway = MTLSGatewayClient(
    server_url=SERVER_WS_URL,
    ca_cert_path=CA_CERT_PATH,
    client_cert_path=CLIENT_CERT_PATH,
    client_key_path=CLIENT_KEY_PATH,
    admission_ctrl=admission,
    require_mtls=REQUIRE_MTLS,
)

print(f"Connecting worker gateway to {SERVER_WS_URL} (Precision: {cluster.precision.upper()})...")
try:
    await gateway.run_forever()
finally:
    orchestrator.shutdown()
    cluster.close()


---
## Mode 3: Interactive Gradio Studio (Web UI + Public Tunnel)
Launches updated Breeze-TTS Studio (`Option A` defaults, auto-gated `.breeze` registration, long-form synthesis, and Whisper Turbo subtitles) on port `7860` with public Gradio URL.

In [ ]:
import glob, os, subprocess, time

model_path = None
if os.path.exists("/kaggle/working/models/active_model.txt"):
    with open("/kaggle/working/models/active_model.txt") as f:
        model_path = f.read().strip()
if not model_path or not os.path.exists(model_path):
    models = sorted(glob.glob("/kaggle/working/models/*.gguf"))
    if not models:
        raise FileNotFoundError("No .gguf model found in /kaggle/working/models. Run Setup cell first.")
    model_path = models[-1]

os.makedirs("/kaggle/working/data/voices", exist_ok=True)
subprocess.run("pkill -9 -f gradio_app.py || true; pkill -9 -f frpc || true", shell=True)
time.sleep(1.5)

log_path = "/kaggle/working/gradio_studio.log"
cmd = (
    "export PYTHONPATH=/kaggle/working/bz-ggml/python PYTHONUNBUFFERED=1 && "
    f"nohup python3 -u /kaggle/working/bz-ggml/python/gradio_app.py "
    f"--model {model_path} "
    f"--lib /kaggle/working/bz-ggml/build/libbreeze.so "
    f"--precision auto "
    f"--vocoder-mode on "
    f"--voices-dir /kaggle/working/data/voices "
    f"--asr-device 0 "
    f"--share --port 7860 > {log_path} 2>&1 &"
)
subprocess.run(cmd, shell=True, executable="/bin/bash", check=True)
print(f"Launching Gradio Studio ({os.path.basename(model_path)}) on port 7860...")

public_url = None
for _ in range(90):
    time.sleep(2)
    if os.path.exists(log_path):
        txt = open(log_path, "r", errors="ignore").read()
        for line in txt.splitlines():
            if "Running on public URL:" in line:
                public_url = line.split("Running on public URL:")[1].strip()
                break
        if public_url:
            break

if not public_url:
    tail = open(log_path).read()[-2000:] if os.path.exists(log_path) else "No log generated"
    raise RuntimeError(f"Gradio Studio did not publish URL in time. Log tail:\n{tail}")

print("=" * 70)
print(f"  PUBLIC GRADIO STUDIO ({os.path.basename(model_path)}): {public_url}")
print("=" * 70)
